# 07 - sklearn Pipelines: Putting It All Together

**Objective:** Learn how to combine preprocessing steps and models into a single sklearn Pipeline.

**Steps:**
1. Basic Pipeline: StandardScaler → LogisticRegression
2. Pipeline with PCA (dimensionality reduction)
3. Pipeline with RFE (feature selection)
4. Custom KMeans feature transformer
5. GridSearchCV over pipeline parameters
6. Save and load the full pipeline
7. Compare pipeline approaches


In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.feature_selection import RFE
from sklearn.cluster import KMeans
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import accuracy_score
from sklearn.base import BaseEstimator, TransformerMixin

import joblib
import warnings
warnings.filterwarnings("ignore")

print("Libraries imported successfully")


Libraries imported successfully


In [ ]:
# TODO: Load the clean data, drop NaNs, separate features and target, and split
# Read data/processed/clean_data.csv, drop any rows with missing values,
# separate X (features) and y (target), then train/test split.
#
# PROCESSED_DIR = Path("../data/processed")
# df = pd.read_csv(PROCESSED_DIR / "clean_data.csv")
# df = df.dropna()
# X = df.drop("class", axis=1).select_dtypes(include=[np.number])
# y = df["class"]
# X_train, X_test, y_train, y_test = train_test_split(
#     X, y, test_size=0.2, stratify=y, random_state=42
# )
# print(f"Train: {X_train.shape}, Test: {X_test.shape}")


### 1. Basic Pipeline: StandardScaler → LogisticRegression

A Pipeline chains multiple transformers and a final estimator. When you call `.fit()`, each step
fits and transforms sequentially (except the last, which only fits). When you call `.predict()`,
each step transforms using the already-fitted parameters — no data leakage.


In [ ]:
# TODO: Build a basic Pipeline that scales features and trains the model
# Chain StandardScaler and the classifier/regressor.
# Fit on X_train, predict on X_test, and print accuracy.
#
# pipe_basic = Pipeline([
#     ("scaler", StandardScaler()),
#     ("clf", LogisticRegression(max_iter=1000, random_state=42)),
# ])
# pipe_basic.fit(X_train, y_train)
# y_pred = pipe_basic.predict(X_test)
# print(f"Basic Pipeline Accuracy: {accuracy_score(y_test, y_pred):.4f}" if not is_reg else f"# print(f"Basic Pipeline RMSE: {np.sqrt(mean_squared_error(y_test, y_pred)):.4f}"
# print(f"\nClassification Report:\n{classification_report(y_test, y_pred, target_names=['cultivar_1', 'cultivar_2', 'cultivar_3'])}")


### 2. Pipeline with PCA

Adding PCA inside the Pipeline means scaling and dimensionality reduction are both fitted on
the training set only, then applied consistently to test data.


In [ ]:
# TODO: Add PCA between scaling and classification
# Insert PCA(n_components=10, random_state=42) after the scaler.
# Fit, predict, and print accuracy + explained variance.
#
# pipe_pca = Pipeline([
#     ("scaler", StandardScaler()),
#     ("pca", PCA(n_components=10, random_state=42)),
#     ("clf", LogisticRegression(max_iter=1000, random_state=42)),
# ])
# pipe_pca.fit(X_train, y_train)
# y_pred_pca = pipe_pca.predict(X_test)
# print(f"PCA Pipeline accuracy: {accuracy_score(y_test, y_pred_pca):.4f}" if not is_reg else f"# print(f"PCA Pipeline RMSE: {{np.sqrt(mean_squared_error(y_test, y_pred_pca)):.4f}}")
# pca = pipe_pca.named_steps["pca"]
# print(f"Explained variance: {pca.explained_variance_ratio_.sum():.2%}")


### 3. Pipeline with RFE

RFE selects the top K features before passing them to the classifier. By embedding RFE in the
Pipeline, feature selection is re-run on each cross-validation fold — no data leakage.


In [ ]:
# TODO: Add RFE for feature selection inside the Pipeline
# Insert RFE(estimator=LogisticRegression(...), n_features_to_select=10) after scaling.
# Print selected features after fitting.
#
# pipe_rfe = Pipeline([
#     ("scaler", StandardScaler()),
#     ("rfe", RFE(estimator=LogisticRegression(max_iter=1000, random_state=42), n_features_to_select=5)),
#     ("clf", LogisticRegression(max_iter=1000, random_state=42)),
# ])
# pipe_rfe.fit(X_train, y_train)
# y_pred_rfe = pipe_rfe.predict(X_test)
# print(f"RFE Pipeline accuracy: {accuracy_score(y_test, y_pred_rfe):.4f}" if not is_reg else f"# print(f"RFE Pipeline RMSE: {{np.sqrt(mean_squared_error(y_test, y_pred_rfe)):.4f}}")
# rfe = pipe_rfe.named_steps["rfe"]
# print(f"Selected features: {list(X.columns[rfe.support_])}")


### 4. Custom KMeans Feature Transformer

You can wrap any sklearn-compatible logic into a custom transformer by implementing
`fit()` and `transform()`. Here we use KMeans to add a cluster label as a new feature,
then combine it with the original features via `FeatureUnion`.


In [ ]:
# TODO: Create a custom transformer and use FeatureUnion
# Implement KMeansFeatureAdder(BaseEstimator, TransformerMixin) with fit/transform
# that appends KMeans cluster labels to the original features.
# Then use FeatureUnion to combine "original" (passthrough) and "cluster".
#
# class KMeansFeatureAdder(BaseEstimator, TransformerMixin):
#     def __init__(self, n_clusters=3, random_state=42):
#         self.n_clusters = n_clusters
#         self.random_state = random_state
#         self.kmeans = KMeans(n_clusters=n_clusters, random_state=random_state, n_init=10)
#
#     def fit(self, X, y=None):
#         self.kmeans.fit(X)
#         return self
#
#     def transform(self, X):
#         clusters = self.kmeans.predict(X).reshape(-1, 1)
#         return np.hstack([X, clusters])
#
# pipe_kmeans = Pipeline([
#     ("scaler", StandardScaler()),
#     ("features", FeatureUnion([
#         ("original", "passthrough"),
#         ("cluster", KMeansFeatureAdder(n_clusters=3)),
#     ])),
#     ("clf", LogisticRegression(max_iter=1000, random_state=42)),
# ])
# pipe_kmeans.fit(X_train, y_train)
# y_pred_kmeans = pipe_kmeans.predict(X_test)
# print(f"KMeans Pipeline accuracy: {accuracy_score(y_test, y_pred_kmeans):.4f}" if not is_reg else f"# print(f"KMeans Pipeline RMSE: {{np.sqrt(mean_squared_error(y_test, y_pred_kmeans)):.4f}}")


### 5. GridSearchCV over Pipeline Parameters

Pipeline parameters are accessed via `step_name__parameter_name` syntax. This lets you tune
preprocessing and model hyperparameters in a single cross-validated search.


In [ ]:
# TODO: Use GridSearchCV to tune pipeline hyperparameters
# Search over pca__n_components and clf__C using the pipe_pca pipeline.
# Use cv=5, scoring='accuracy'.
#
# param_grid = {
#     "pca__n_components": [5, 10, 15, 20],
#     "clf__C": [0.01, 0.1, 1.0, 10.0],
# }
# grid = GridSearchCV(pipe_pca, param_grid, cv=5, scoring="accuracy", n_jobs=-1)
# grid.fit(X_train, y_train)
# print(f"Best CV accuracy: {grid.best_score_:.4f}")
# print(f"Best params: {grid.best_params_}")


### 6. Save and Load the Full Pipeline

The entire pipeline — scaler, PCA, and classifier — is saved as a single `joblib` file.
When loaded, it applies all preprocessing automatically before predicting. This eliminates
hardcoded feature names and manual transform steps.


In [ ]:
# TODO: Save the full pipeline and reload for inference
# Use joblib.dump() to save grid.best_estimator_ or pipe_basic.
# Then load it back and predict on X_test to verify it works.
#
# MODELS_DIR = Path("../models")
# MODELS_DIR.mkdir(parents=True, exist_ok=True)
#
# joblib.dump(grid.best_estimator_, MODELS_DIR / "pipeline_model.joblib")
# print("Pipeline saved!")
#
# loaded = joblib.load(MODELS_DIR / "pipeline_model.joblib")
# print(f"Loaded pipeline accuracy: {accuracy_score(y_test, loaded.predict(X_test)):.4f}" if not is_reg else f"# print(f"Loaded pipeline RMSE: {{np.sqrt(mean_squared_error(y_test, loaded.predict(X_test))) :.4f}}")


### 7. Comparison

All pipelines use the same training/testing data. The table below compares their metrics:


In [ ]:
# TODO: Create a comparison table of pipeline metrics
# Store each pipeline's accuracy/rmse and display as a DataFrame.
#
# comparison = pd.DataFrame({
#     "Pipeline": ["Scaler → LR", "Scaler → PCA → LR", "Scaler → RFE → LR"],
#     "Accuracy": [acc_basic, acc_pca, acc_rfe],
# })
# print(comparison.to_string(index=False))


### Exercises

1. **Try different classifiers**: Replace with RandomForest or XGBoost in the pipeline.
2. **Add more PCA components**: Change `n_components` in the PCA pipeline.
3. **Pipeline for feature scaling only**: Create a pipeline with KNN. Compare unscaled.
4. **Nested cross-validation**: Use `cross_val_score` with `cv=10` for robust estimates.
5. **Custom transformer**: Write a transformer using `VarianceThreshold`.
6. **ColumnTransformer**: For mixed types, apply different preprocessing per column.
